# Cityscapes Semantic Segmentation 

Urban autonomous systems rely on accurate perception of the surrounding environment. Applications such as last‑mile delivery, traffic monitoring, route planning, and autonomous transport require detailed spatial understanding of urban scenes—often at pixel level.

In this exercise, we work with the **Cityscapes** dataset, a large-scale dataset of high-resolution street-level images captured in real traffic in multiple European cities. Each image is annotated with pixel-wise semantic labels, making Cityscapes a standard benchmark for semantic segmentation in urban environments.

### What we do in this notebook
You will build a semantic segmentation pipeline in **PyTorch** that:
1. loads and inspects Cityscapes images and semantic masks,
2. simplifies the label space to 4 classes relevant to transportation,
3. adapts the experiment to CPU-only hardware (subset + resize),
4. fine-tunes a pretrained **DeepLabV3** model,
5. evaluates performance using **Pixel Accuracy** and **mean IoU**,
6. visualizes predictions and typical failure cases.



## Downloading the Cityscapes dataset 
The **Cityscapes dataset** is a large-scale benchmark for semantic understanding of urban street scenes. It consists of high-resolution RGB images captured from a vehicle driving through multiple European cities, together with pixel-wise semantic segmentation annotations for a wide range of urban object classes such as roads, buildings, vehicles, pedestrians, and traffic infrastructure. Cityscapes is widely used in research on autonomous driving, intelligent transportation systems, and urban scene analysis.

Follow the steps below to obtain the required data.

### Step 1: Create a Cityscapes account
Cityscapes data is available only to registered users.

1. Go to the official Cityscapes website:  
   https://www.cityscapes-dataset.com/
2. Create an account and log in.
3. Accept the dataset license agreement.


### Step 2: Download the required archives
You only need **two components** of Cityscapes for this exercise:

- **Left RGB images**  
  `leftImg8bit_trainvaltest.zip`

- **Fine semantic annotations**  
  `gtFine_trainvaltest.zip`

These contain:
- RGB images in `leftImg8bit/`
- Semantic label masks in `gtFine/`

You do **not** need:
- coarse annotations (`gtCoarse`)
- disparity maps
- stereo right images
- extra training sets

### Step 3: Extract the archives
After downloading, extract both ZIP files into the same root directory:



Please install Pytorch and torchvision before running the notebook. The Jupyter Notebook depends on the PyTorch / torchvision version, among other things. Please make sure that your version is up to date.

Now import Pytorch and torchvision and check the versions of the two libraries.

In [ ]:
import torch, torchvision

# print versions

## Task 1: Creating the basic framework

### Task 1.1: Importing modules

As a first step, load the libraries for analyzing and manipulating data presented in the lecture, as well as PyTorch and torchvision. Refer to this cheatsheet fora quick introduction to common modules that are used while using Pytorch - https://pytorch-cn.com/tutorials/beginner/ptcheat.html

In [ ]:
# TODO: Import the required modules.
# Hints: os, Path, random, numpy, matplotlib, torch, nn, DataLoader, torchvision, transforms as T, IPython.display



import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms as T



### Task 1.2: Reproducibility and device

Set a random seed so that results are reproducible and choose CPU/GPU.

In [ ]:
def set_seed(seed: int = 42):
    """
    Complete set_seed so that results are reproducible across runs.
    """
    random.seed(seed)
    np.random.seed(seed)
    # TODO: set torch seeds (CPU + CUDA)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # TODO: set cudnn flags to deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

# TODO: choose device (cuda if available else cpu)


device

## Task 2: Loading and understanding the Cityscapes dataset

### Task 2.1: Dataset location

After downloading, you will have a folder that contains (among others) the subfolders:
- `leftImg8bit/`
- `gtFine/` (or `gtCoarse/`)

Set the path to the **root folder** of Cityscapes below.

<span style="color:blue">Execute the following cell. If it fails, adjust the path accordingly.</span>


In [ ]:
# TODO: Update this path to your local Cityscapes root directory
CITYSCAPES_ROOT = Path(" ./cityscapes")

# TODO: Verify the folder structure contains 'leftImg8bit' and 'gtFine'
print("Expecting:", CITYSCAPES_ROOT / "leftImg8bit")
print("Expecting:", CITYSCAPES_ROOT / "gtFine")

assert (CITYSCAPES_ROOT / "leftImg8bit").exists(), "leftImg8bit folder not found. Check CITYSCAPES_ROOT."
assert (CITYSCAPES_ROOT / "gtFine").exists(), "gtFine folder not found. Check CITYSCAPES_ROOT."

CITYSCAPES_ROOT

### Task 2.2: Editing the dataset further

Since we do not have many training resources and the downloaded dataset is still large, we need to reduce it further. To do this, we retain data for only certain cities both for training and validation folders 

In [ ]:
import shutil

# Keep only these cities
KEEP_CITIES = {"hamburg", "hanover"}

train_img_root = CITYSCAPES_ROOT / "leftImg8bit" / "train"
train_lbl_root = CITYSCAPES_ROOT / "gtFine" / "train"


def prune_city_folders(root_dir: Path, keep_cities: set):
    for city_dir in root_dir.iterdir():
        if not city_dir.is_dir():
            continue
        if city_dir.name.lower() not in keep_cities:
            print(f"Deleting {city_dir}")
            shutil.rmtree(city_dir)


print("Pruning image folders...")
prune_city_folders(train_img_root, KEEP_CITIES)

print("Pruning label folders...")
prune_city_folders(train_lbl_root, KEEP_CITIES)

print("Done.")

<span style="color:blue">Modify the code above so that, in the validation split, all city folders are removed except Frankfurt..</span>

### Background

Before we can train a deep learning model, we must first define how the data is presented to the model. In PyTorch, this is done by creating dataset objects that specify how raw data on disk is loaded, interpreted, and returned to the training pipeline.
In this step, we use `torchvision.datasets.Cityscapes` to create dataset objects for the training and validation splits. Each call to the dataset returns an RGB image as a PIL object, and a corresponding semantic segmentation mask, where each pixel encodes a class label.

We explicitly select `target_type='semantic'` to ensure that the dataset returns pixel-wise class annotations, which are required for semantic segmentation. These masks serve as the ground truth that the model learns to predict.This step is crucial because it defines the input–output relationship the model will learn from, and it ensures that images and labels are correctly aligned


### Task 2.3: Creating a dataset object

`torchvision.datasets.Cityscapes` returns:
- the RGB image,
- the corresponding target (mask / polygon, depending on `target_type`).

We will use **semantic masks** from `gtFine` via `target_type='semantic'`.

<span style="color:blue">Create the dataset object and inspect its length.</span>

In [ ]:
# Basic transforms for the input image
img_transform = T.Compose([
    T.ToTensor(),
    # Normalization for ImageNet-pretrained backbones:
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_city = torchvision.datasets.Cityscapes(
    root=str(CITYSCAPES_ROOT),
    split="train",
    mode="fine",
    target_type="semantic",
    transform=None,
    target_transform=None
)

val_city = torchvision.datasets.Cityscapes(
    root=str(CITYSCAPES_ROOT),
    split="val",
    mode="fine",
    target_type="semantic",
    transform=None,
    target_transform=None
)

len(train_city), len(val_city)

### Task 2.4: Visualizing samples

<span style="color:blue">Execute the cell and interpret the result.</span>


In [ ]:
def show_image_and_mask_pil(dataset, idx: int = 0):
    img, mask = dataset[idx]  # PIL, PIL
    print("Image:", img.mode, img.size, "| Mask:", mask.mode, mask.size)
    display(img)
    display(mask)
    return img, mask


_ = show_image_and_mask_pil(train_city, idx=0)

 <span style="color:blue">Write your own visualization helper that overlays the mask on the RGB image and helps in a better visualization of the classes.</span>

In [ ]:
# enter the code here.

### Task 2.4: Mapping Cityscapes ids to a small set of classes
The original Cityscapes semantic segmentation masks contain many fine-grained classes, each represented by a unique integer ID (e.g., road, sidewalk, building, car, bus, bicycle, etc.). Training a model to predict all original classes increases both computational cost and learning complexity.

In this step, we simplify the segmentation problem by grouping several original Cityscapes classes into a smaller set of **four high-level categories**:
- `0`: background / all other classes  
- `1`: road  
- `2`: sidewalk  
- `3`: vehicles (car, truck, bus, train, motorcycle, bicycle)

<span style="color:blue">The function `remap_mask_to_4_classes` takes an original Cityscapes mask and remaps pixel values from the original class IDs to this reduced label set. Pixels belonging to road, sidewalk, or vehicle-related classes are assigned new integer labels, while all remaining pixels are treated as background.</span>

<span style="color:blue">Complete this function </span> 


In [ ]:
ROAD_IDS = {7}
SIDEWALK_IDS = {8}

VEHICLE_IDS = {26, 27, 28, 31, 32, 33}


def remap_mask_to_4_classes(mask_np: np.ndarray) -> np.ndarray:
    out = np.zeros_like(mask_np, dtype=np.uint8)  # 0 background
    # TODO: set road pixels to 1
   
    # TODO: set sidewalk pixels to 2
    
    # TODO: set vehicle pixels to 3
   
    return out


test = np.array([[0, 7, 8, 26, 33]], dtype=np.uint8)
remap_mask_to_4_classes(test)

### Task 2.5: Image resizing and subset selection

This code defines a helper function that resizes an image and its corresponding segmentation mask in a consistent way. Resizing is a common preprocessing step that reduces computational cost and ensures that all samples have the same spatial resolution.The function `resize_pair` takes two inputs: an RGB image (`img_pil`) and its corresponding segmentation mask (`mask_pil`), both provided as PIL images. The target resolution is specified by `size_hw` as `(height, width)`.
By resizing the image and mask together using the same target resolution, this function preserves their pixel-wise alignment, which is essential for semantic segmentation. Incorrect or inconsistent resizing is a common source of subtle bugs in segmentation pipelines.


<span style="color:blue"> Write the code to resize the images and masks. Two different interpolation methods are supposed to be used while resizing the images and masks respectively. Why is this the case?</span>

In [ ]:
# Choose a smaller training resolution (H, W).
RESIZE_HW = (256, 512)

import torchvision.transforms.functional as F
from torchvision.transforms import InterpolationMode


def resize_pair(img_pil, mask_pil, size_hw):
    # ToDo: resize image with bilinear interpolation mode
    
    # ToDo: resize mask with nearest interpolation mode
   
    return img_r, mask_r

### Task 2.6: Building a wrapped dataset

 <span style="color:blue">Instantiate `train_ds` and `val_ds`using `Cityscapes4Class` by loading one sample from the training dataset. Check the shapes of the image and mask tensors and inspect the unique values in the masks.</span>

Confirm that: image and mask spatial dimensions match, the image tensor has the correct format and mask values belong to the reduced 4-class label space.
    
Do not proceed to training until these checks pass.


In [ ]:
class Cityscapes4Class(torch.utils.data.Dataset):
    """
    Wraps a torchvision Cityscapes dataset to:
      1) optionally resize (CPU-friendly) while keeping image/mask aligned,
      2) apply an image transform (ToTensor + Normalize),
      3) remap Cityscapes labelIds to a reduced 4-class mask,
      4) return (x, y) as tensors suitable for training.
    """

    def __init__(self, base_dataset, img_transform, resize_hw=None):
        self.base = base_dataset
        self.img_transform = img_transform
        self.resize_hw = resize_hw

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]  # PIL image, PIL mask (labelIds)

        # Optional sanity check (raw data)
        if img.size != mask.size:
            raise ValueError(
                f"Image/mask size mismatch at idx={idx}: image={img.size}, mask={mask.size}"
            )

        # Optional resize for CPU friendliness (must resize both!)
        if self.resize_hw is not None:
            img, mask = resize_pair(img, mask, self.resize_hw)

        # Transform image to normalized tensor [3, H, W]
        x = self.img_transform(img)

        # Convert mask to numpy labelIds, remap to 4 classes, then to tensor [H, W]
        mask_np = np.array(mask, dtype=np.uint8)
        mask_4 = remap_mask_to_4_classes(mask_np)   # expects uint8 labelIds
        y = torch.from_numpy(mask_4).long()

        return x, y

In [ ]:
# Instantiate and verify the training and validation datsets here.


### Task 2.7: Subset selection and experimentation

In this task, you will experiment with different subset sizes and observe how they influence training time, convergence behavior, and validation performance (Pixel Accuracy and mean IoU). You are encouraged to try at least two different configurations rather than following a single fixed choice.

Typical configurations might include, for example, 100 training images with 50 validation images, 200 training images with 50 validation images, or 300 training images with 100 validation images. You are not limited to these values and may choose others that suit your hardware.

While experimenting, consider how increasing the number of training samples affects convergence stability and validation metrics, and at what point training becomes impractically slow on your machine. Smaller subsets usually train faster but can lead to noisier or less reliable validation results, whereas larger subsets often improve generalization at the cost of increased computation time.


In [ ]:
from torch.utils.data import Subset

N_TRAIN = 200
N_VAL = 50

train_ds = Subset(train_ds, range(N_TRAIN))
val_ds = Subset(val_ds, range(N_VAL))

len(train_ds), len(val_ds)

### Task 2.8: DataLoaders

<span style="color:blue">Choose a batch size that works on your machine (start with 1–4 on CPU).</span>


In [ ]:
BATCH_SIZE = 2  

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

next(iter(train_loader))[0].shape

## Task 3: Defining a segmentation model (DeepLabV3)

### Task 3.1: Loading a pretrained model

We use `torchvision.models.segmentation.deeplabv3_resnet50` with ImageNet-pretrained backbone. We replace the classifier head to output **4 classes**.

<span style="color:blue">
Fill in the missing code and execute the following cell. What is transfer learning and why do we use it here?
</span>


In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)

# ToDo: Replace the last layer of model.classifier so it outputs 4 classes.


model = model.to(device)

sum(p.numel() for p in model.parameters()) / 1e6

### Task 3.2: (Optional) Freeze the backbone (CPU speed-up)

In [ ]:
# TODO (optional): Freeze backbone parameters to speed up CPU training.
# for p in model.backbone.parameters():
#     p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
total = sum(p.numel() for p in model.parameters()) / 1e6
trainable, total

### Task 3.3: Loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)

criterion, optimizer

## Task 4: Training and evaluation

### Task 4.0 : Limit training to a fixed number of batches (CPU)

In [ ]:
MAX_BATCHES_PER_EPOCH = 50

### Task 4.1: Evaluation metrics for semantic segmentation

To assess the quality of a semantic segmentation model, we need evaluation metrics that compare the predicted class label of each pixel with the corresponding ground-truth label. In this exercise, we use two commonly applied metrics: **Pixel Accuracy** and **mean Intersection over Union (mIoU)**. Both operate at the pixel level but capture different aspects of model performance.

#### Pixel Accuracy

Pixel Accuracy measures the fraction of correctly classified pixels over the entire image (or dataset). It is computed as the number of pixels for which the predicted label matches the ground truth, divided by the total number of pixels.

#### Mean Intersection over Union (mIoU)
Mean Intersection over Union is a more informative metric for semantic segmentation. For each class, it computes the ratio between:
- the intersection: pixels correctly predicted as belonging to that class, and
- the union: all pixels that are predicted as that class or belong to that class in the ground truth.

The IoU is computed per class, and the final score is obtained by averaging over all classes (hence *mean* IoU). Classes that do not appear in either prediction or ground truth are skipped.



<span style="color:blue">
Execute the cell and read the functions.  
Which metric is more informative when classes are imbalanced?
</span>



In [ ]:
@torch.no_grad()
def pixel_accuracy(pred: torch.Tensor, target: torch.Tensor) -> float:
    correct = (pred == target).float().sum().item()
    total = torch.numel(target)
    return correct / total


@torch.no_grad()
def mean_iou(pred: torch.Tensor, target: torch.Tensor, num_classes: int = 4, eps: float = 1e-6) -> float:
    ious = []
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)

        inter = (pred_c & target_c).float().sum().item()
        union = (pred_c | target_c).float().sum().item()

        if union == 0:
            continue
        ious.append((inter + eps) / (union + eps))
    return float(np.mean(ious)) if len(ious) > 0 else 0.0

### Task 4.2: One training epoch

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    batch_idx = 0
    n = 0

    for x, y in loader:
        if MAX_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_BATCHES_PER_EPOCH:
            break
        batch_idx += 1

        x = x.to(device)
        y = y.to(device)

        out = model(x)["out"]
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

    return total_loss / max(1, n)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    accs = []
    mious = []
    n = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        out = model(x)["out"]
        loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)

        pred = out.argmax(dim=1)
        accs.append(pixel_accuracy(pred, y))
        mious.append(mean_iou(pred, y, num_classes=4))

    return (total_loss / max(1, n), float(np.mean(accs)), float(np.mean(mious)))

### Task 4.3a: Training loop

In [ ]:
EPOCHS = 2

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_miou": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc, va_miou = validate(model, val_loader, criterion, device)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["val_miou"].append(va_miou)

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_acc={va_acc:.4f} | val_mIoU={va_miou:.4f}")

### Task 4.3b: Hyperparameter experiments

Try **at least two modifications** to the training setup and write down what changed in terms of **training loss** and **validation performance (mIoU)**.

You may experiment with any of the following hyperparameters (examples only):

- learning rate: `1e-4 → 3e-4` or `1e-4 → 1e-5`  
- weight decay: `1e-4 → 0`  
- batch size: `2 → 4`  
- number of epochs: `2 → 5`  

Use the empty cells below to **document your experiments**, including:
- what you changed,
- why you changed it,
- and how it affected training stability and performance.


In [ ]:
# Experiment 1



In [ ]:
# Experiment 2



### Task 4.4: Plot learning curves

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

### Task 4.5: Visualize predictions

In [ ]:
def unnormalize(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device).view(3,1,1)
    return (img_tensor * std + mean).clamp(0, 1)


@torch.no_grad()
def show_prediction(model, dataset, idx=0):
    model.eval()
    x, y = dataset[idx]
    x = x.to(device)
    out = model(x.unsqueeze(0))["out"]
    pred = out.argmax(dim=1).squeeze(0).cpu()

    x_vis = unnormalize(x).permute(1,2,0).cpu().numpy()
    y_np = y.cpu().numpy()
    p_np = pred.numpy()

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(x_vis)
    ax[0].set_title("Input image")
    ax[0].axis("off")

    ax[1].imshow(y_np, vmin=0, vmax=3)
    ax[1].set_title("Ground truth (4 classes)")
    ax[1].axis("off")

    ax[2].imshow(p_np, vmin=0, vmax=3)
    ax[2].set_title("Prediction (4 classes)")
    ax[2].axis("off")

    plt.show()
    plt.close(fig)


show_prediction(model, val_ds, idx=0)

## Task 5: Discussion 

1. **Class imbalance:** Road pixels dominate. Why can Pixel Accuracy be misleading?
2. **Better metrics:** Why is mean IoU commonly used for segmentation benchmarks?
3. **Possible improvements:**
   - train longer + learning rate schedule,
   - stronger augmentations (random crop, color jitter),
   - class-weighted loss,
   - train on full Cityscapes label set (19 classes) using `trainId`.


**Reference**

M. Cordts, M. Omran, S. Ramos, T. Rehfeld, M. Enzweiler, R. Benenson,  U. Franke, S. Roth, and B. Schiele,  *The Cityscapes Dataset for Semantic Urban Scene Understanding*,  Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2016.



   <a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons Lizenzvertrag" style="border-width:0; display:inline" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a> &nbsp;&nbsp;&nbsp;&nbsp;This work by Shubhangi Gupta is licensed under a  <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.
